# Customer Churn Prediction - Inference

Use your fine-tuned GPT-3.5 Turbo model to predict customer churn risk.

## Prerequisites
- Fine-tuned model ID saved in environment: `CUSTOMER_CHURN_OPEN_AI_MODEL`
- OpenAI API key: `OPEN_AI_FINE_TUNING_KEY`

## 1. Setup

In [ ]:
%pip install openai -q

In [ ]:
import os
import json
from openai import OpenAI
from IPython import get_ipython

# Load API key
api_key = get_ipython().getoutput('echo $OPEN_AI_FINE_TUNING_KEY')
if api_key and api_key[0].strip():
    os.environ['OPEN_AI_FINE_TUNING_KEY'] = api_key[0].strip()
else:
    raise ValueError("❌ OPEN_AI_FINE_TUNING_KEY not found")

# Load model ID
model_id = get_ipython().getoutput('echo $CUSTOMER_CHURN_OPEN_AI_MODEL')
if model_id and model_id[0].strip():
    FINE_TUNED_MODEL = model_id[0].strip()
    print(f"✅ Model loaded: {FINE_TUNED_MODEL}")
else:
    raise ValueError("❌ CUSTOMER_CHURN_OPEN_AI_MODEL not found")

client = OpenAI(api_key=os.getenv('OPEN_AI_FINE_TUNING_KEY'))
print("✅ Client initialized")

## 2. Single Prediction

In [ ]:
# Test with a single customer
test_prompt = "Customer bought 500ml regularly for 2 months. Price increased by 15% recently. They raised 2 complaint(s) about product defect. Last purchase was 40 days ago. Competitor has reduced their prices recently. Predict churn risk."

print("📝 Customer Profile:")
print(f"   {test_prompt}\n")

response = client.chat.completions.create(
    model=FINE_TUNED_MODEL,
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.7,
    max_tokens=200
)

print("🤖 Prediction:")
print(response.choices[0].message.content)
print(f"\n💰 Tokens used: {response.usage.total_tokens}")

## 3. Batch Predictions

In [ ]:
def predict_churn(customer_profile: str) -> dict:
    """Predict churn for a single customer profile."""
    prompt = f"{customer_profile} Predict churn risk."
    
    try:
        response = client.chat.completions.create(
            model=FINE_TUNED_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=200
        )
        
        return {
            "success": True,
            "prediction": response.choices[0].message.content,
            "tokens": response.usage.total_tokens
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


# Test batch
test_customers = [
    "Customer bought 1000ml regularly for 7 months. Price increased by 3% recently. No complaints. Last purchase was 5 days ago.",
    "Customer bought 250ml slightly irregularly for 4 months. Price increased by 10% recently. They raised 1 complaint(s) about delivery delay. Last purchase was 25 days ago.",
    "Customer bought 500ml irregularly for 1 month. Price increased by 18% recently. They raised 3 complaint(s) about multiple issues. Last purchase was 42 days ago. Competitor has reduced their prices recently."
]

print("🔮 Batch Predictions\n")
print("=" * 80)

total_tokens = 0
for i, customer in enumerate(test_customers, 1):
    print(f"\n📋 Customer {i}:")
    print(f"   {customer}")
    
    result = predict_churn(customer)
    if result["success"]:
        print(f"\n   Prediction:")
        print(f"   {result['prediction']}")
        print(f"   Tokens: {result['tokens']}")
        total_tokens += result['tokens']
    else:
        print(f"   ❌ Error: {result['error']}")
    
    print("\n" + "-" * 80)

print(f"\n💰 Total tokens used: {total_tokens}")

## 4. Compare with Base Model

In [ ]:
test_prompt = "Customer bought 500ml regularly for 2 months. Price increased by 15% recently. They raised 2 complaint(s) about product defect. Last purchase was 40 days ago. Competitor has reduced their prices recently. Predict churn risk."

print("📊 Model Comparison\n")
print("=" * 70)

# Base model
print("\n🔹 Base GPT-3.5 Turbo:")
base_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.7,
    max_tokens=200
)
print(base_response.choices[0].message.content)

print("\n" + "=" * 70)

# Fine-tuned model
print("\n✨ Fine-tuned Model:")
fine_tuned_response = client.chat.completions.create(
    model=FINE_TUNED_MODEL,
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.7,
    max_tokens=200
)
print(fine_tuned_response.choices[0].message.content)

print("\n" + "=" * 70)
print("\n💡 Notice: Fine-tuned model provides structured output with consistent format.")

## 5. Interactive Prediction

In [ ]:
# Enter your own customer profile
custom_profile = input("Enter customer profile (or press Enter for example): ").strip()

if not custom_profile:
    custom_profile = "Customer bought 250ml irregularly for 3 months. Price increased by 12% recently. Last purchase was 30 days ago."
    print(f"Using example: {custom_profile}\n")

result = predict_churn(custom_profile)

if result["success"]:
    print("\n🤖 Prediction:")
    print(result['prediction'])
else:
    print(f"\n❌ Error: {result['error']}")

## 6. Export Predictions to JSON

In [ ]:
# Batch predict and save to file
customers = [
    "Customer bought 1000ml regularly for 7 months. Price increased by 3% recently. No complaints. Last purchase was 5 days ago.",
    "Customer bought 500ml irregularly for 1 month. Price increased by 18% recently. They raised 3 complaint(s). Last purchase was 42 days ago."
]

results = []
for customer in customers:
    prediction = predict_churn(customer)
    results.append({
        "customer_profile": customer,
        "prediction": prediction.get("prediction", "Error"),
        "success": prediction["success"]
    })

# Save to JSON
output_file = "churn_predictions.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Predictions saved to {output_file}")
print(f"   Total predictions: {len(results)}")